# Expectation Values and Hamiltonians

This tutorial covers how to compute expectation values of quantum observables using qiskit-trev, including:

- Defining Hamiltonians with `SparsePauliOp`
- Using `TREVEstimator` (Qiskit primitive)
- Using `TensorRingModel` directly
- Measurement methods: efficient contraction vs full contraction

In [ ]:
import math
import torch
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_trev import TREVEstimator, TensorRingModel

## 1. Defining Hamiltonians

Observables are defined using Qiskit's `SparsePauliOp`. Each term is a tensor product of Pauli operators (I, X, Y, Z) with a coefficient.

**Pauli string ordering:** Qiskit uses little-endian qubit ordering. `"ZI"` means Z on qubit 1 and I on qubit 0.

In [ ]:
# Simple: single Z operator on a 1-qubit system
H_simple = SparsePauliOp.from_list([("Z", 1.0)])

# Ising-like: ZZ interactions + transverse field
H_ising = SparsePauliOp.from_list([
    ("ZZI", 1.0),   # ZZ on qubits 2,1
    ("IZZ", 1.0),   # ZZ on qubits 1,0
    ("XII", -0.5),   # X field on qubit 2
    ("IXI", -0.5),   # X field on qubit 1
    ("IIX", -0.5),   # X field on qubit 0
])

print("Simple H:", H_simple)
print("Ising H:", H_ising)

## 2. TREVEstimator (Qiskit Primitive)

`TREVEstimator` implements Qiskit's `BaseEstimatorV2`. It accepts standard Qiskit circuits and observables, making it a drop-in replacement for other estimators.

In [ ]:
# Prepare |+> state: H|0>
qc = QuantumCircuit(1)
qc.h(0)

estimator = TREVEstimator(rank=4, device="cpu")

# <+|Z|+> = 0, <+|X|+> = 1
H_z = SparsePauliOp.from_list([("Z", 1.0)])
H_x = SparsePauliOp.from_list([("X", 1.0)])

result_z = estimator.run([(qc, H_z)]).result()
result_x = estimator.run([(qc, H_x)]).result()

print(f"<+|Z|+> = {result_z[0].data.evs:.4f}  (expected 0.0)")
print(f"<+|X|+> = {result_x[0].data.evs:.4f}  (expected 1.0)")

## 3. TensorRingModel for Parameterized Circuits

For variational workflows, `TensorRingModel` is more efficient. It pre-compiles the circuit structure and supports batched evaluation.

In [ ]:
# 2-qubit parameterized circuit
qc = QuantumCircuit(2)
qc.ry(0.0, 0)
qc.ry(0.0, 1)
qc.cx(0, 1)

# ZZ + ZI Hamiltonian
H = SparsePauliOp.from_list([("ZZ", 1.0), ("ZI", 0.5)])

model = TensorRingModel(qc, H, rank=4, device="cpu")

# Single evaluation
theta = torch.tensor([0.3, 0.7])
ev = model(theta)
print(f"<H>(theta=[0.3, 0.7]) = {ev.item():.4f}")

# Batched evaluation: 5 different parameter sets at once
params_batch = torch.randn(5, 2)
evs = model.evaluate_batch(params_batch)
print(f"\nBatched results for 5 parameter sets:")
for i in range(5):
    print(f"  params={params_batch[i].tolist()}  <H>={evs[i].item():.4f}")

## 4. Measurement Methods

qiskit-trev provides two contraction strategies:

- **Efficient contraction** — uses double-layer transfer matrices. Natively handles Z/I terms. For X/Y terms, it first rotates the tensor ring into the appropriate measurement basis, then contracts in the Z basis. Scales as O(N * chi^4) per term. Works for large N.
- **Full contraction** — contracts the tensor ring to a full 2^N statevector, then computes `<psi|H|psi>` directly. Handles any Hamiltonian but limited to ~20 qubits.

You can call either method directly via the functions in `qiskit_trev.measure`. `TensorRingModel` currently auto-selects: efficient contraction for Z/I-only Hamiltonians, full contraction when X/Y terms are present.

In [ ]:
from qiskit_trev.measure.efficient_contraction import expectation_value as ev_efficient
from qiskit_trev.measure.full_contraction import expectation_value as ev_full
from qiskit_trev.hamiltonian import rotate_tensor_for_measurement
from qiskit_trev import TensorRingState, circuit_to_gate_instructions, sparse_pauli_op_to_hamiltonian

# Build a Bell state tensor ring
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
gates, _ = circuit_to_gate_instructions(qc)
state = TensorRingState(2, rank=4, device="cpu")
tensor = state.build(gates)

# --- Efficient contraction: Z/I terms (no rotation needed) ---
H_zz = sparse_pauli_op_to_hamiltonian(SparsePauliOp.from_list([("ZZ", 1.0)]))
ev = ev_efficient(tensor, H_zz)
print(f"Efficient contraction <Bell|ZZ|Bell> = {ev:.4f}")

# --- Full contraction: works with any Hamiltonian ---
H_xx = sparse_pauli_op_to_hamiltonian(SparsePauliOp.from_list([("XX", 1.0)]))
ev = ev_full(tensor, H_xx)
print(f"Full contraction      <Bell|XX|Bell> = {ev:.4f}")

# --- Efficient contraction with X terms via basis rotation ---
# Group by qubit-wise commuting (QWC) bases, rotate, then use efficient contraction
groups = H_xx.get_qwc_groups()
print(f"\nQWC groups for XX: {groups}")
basis = groups[0]['basis']  # "XX"
rotated = rotate_tensor_for_measurement(tensor, basis)
# After rotating into X basis, measure in Z basis
H_zz_proxy = sparse_pauli_op_to_hamiltonian(SparsePauliOp.from_list([("ZZ", 1.0)]))
ev = ev_efficient(rotated, H_zz_proxy)
print(f"Efficient + rotation  <Bell|XX|Bell> = {ev:.4f}")

## 5. Computing Gradients

`parameter_shift_grad` computes exact analytic gradients using the parameter-shift rule. This is fully batched — all 2P shifted circuits are evaluated in one GPU call.

In [ ]:
# RY(theta)|0> with Z observable: <Z> = cos(theta), d<Z>/dtheta = -sin(theta)
qc = QuantumCircuit(1)
qc.ry(0.0, 0)
model = TensorRingModel(qc, SparsePauliOp.from_list([("Z", 1.0)]), rank=1, device="cpu")

theta = torch.tensor([0.7])
grad = model.parameter_shift_grad(theta)
print(f"theta = {theta.item():.4f}")
print(f"  gradient     = {grad[0].item():.4f}")
print(f"  -sin(theta)  = {-math.sin(theta.item()):.4f}")